### **MNIST**

In [18]:
import numpy as np
from library.gmm import GMM
from library.knn import KNN
from library.lda import LDA
from library.pca import PCA

### **Load Data**

In [19]:
import os
from PIL import Image

def load_images(folder):
    images = []
    labels = []
    for digit in [0, 1, 7]:
        digit_folder = f"{folder}/{digit}"
        for filename in sorted(os.listdir(digit_folder)):
            if filename.endswith(".png"):
                with Image.open(f"{digit_folder}/{filename}") as image:
                    pixels = np.array(image.convert("L"), dtype=float)
                    images.append(pixels.ravel() / 255.0)
                    labels.append(digit)
    return np.array(images), np.array(labels)

X_train, y_train = load_images("./data/mnist/train")
X_val, y_val = load_images("./data/mnist/dev")
X_test, y_test = load_images("./data/mnist/test")

classes = np.unique(y_train)
print("Class order:", classes)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Class order: [0 1 7]
Train: (15101, 784)
Validation: (3829, 784)
Test: (3143, 784)


### **GMM**

In [20]:
models = []
priors = []
for label in classes:
    samples = X_train[y_train == label]
    model = GMM(n_components=3, regularization=1e-3, iterations=100, convergence_tol=1e-4)
    model.fit(samples)
    models.append(model)
    priors.append(len(samples) / len(X_train))
    print("Class", label, "EM iterations:", len(model.log_likelihoods))
    if len(model.log_likelihoods) == model.iterations:
        print("Iteration limit reached; convergence is not guaranteed.")

val_scores = np.column_stack([model.score_samples(X_val) for model in models])
gmm_val_predictions = classes[np.argmax(val_scores + np.log(priors), axis=1)]

test_scores = np.column_stack([model.score_samples(X_test) for model in models])
gmm_test_predictions = classes[np.argmax(test_scores + np.log(priors), axis=1)]

# Confusion matrix
gmm_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_confusion)

Class 0 EM iterations: 19
Class 1 EM iterations: 49
Class 7 EM iterations: 24
Validation accuracy: 0.9869417602507182
Test accuracy: 0.9895004772510341
Test confusion matrix:
 [[ 977    0    3]
 [  17 1117    1]
 [  10    2 1016]]


### **KNN**

In [21]:
knn = KNN(neighbors=5)
knn.fit(X_train, y_train)
knn_val_predictions = knn.predict(X_val)
knn_test_predictions = knn.predict(X_test)

# Confusion matrix
knn_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_test_predictions == y_test))
print("Test confusion matrix:\n", knn_confusion)

Validation accuracy: 0.9942543745103161
Test accuracy: 0.9904549793191219
Test confusion matrix:
 [[ 976    2    2]
 [   0 1135    0]
 [   0   26 1002]]


### **LDA**

In [22]:
lda = LDA()
lda.fit(X_train, y_train)
lda_val_predictions = lda.predict(X_val)
lda_test_predictions = lda.predict(X_test)

# Confusion matrix
lda_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        lda_confusion[i, j] = np.sum(
            (y_test == actual) & (lda_test_predictions == predicted))

print("Validation accuracy:", np.mean(lda_val_predictions == y_val))
print("Test accuracy:", np.mean(lda_test_predictions == y_test))
print("Test confusion matrix:\n", lda_confusion)

Validation accuracy: 0.9845912770958475
Test accuracy: 0.9853643016226535
Test confusion matrix:
 [[ 970    2    8]
 [   1 1131    3]
 [   4   28  996]]


### **PCA**

In [23]:
K = 50
pca = PCA(n_components=K)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca = pca.transform(X_test)

print("Train:", X_train_pca.shape)
print("Validation:", X_val_pca.shape)
print("Test:", X_test_pca.shape)

Train: (15101, 50)
Validation: (3829, 50)
Test: (3143, 50)


### **GMM with PCA**

In [24]:
pca_models = []
pca_priors = []
for label in classes:
    pca_samples = X_train_pca[y_train == label]
    pca_model = GMM(n_components=3, regularization=1e-3, iterations=100, convergence_tol=1e-4)
    pca_model.fit(pca_samples)
    pca_models.append(pca_model)
    pca_priors.append(len(pca_samples) / len(X_train_pca))
    print("Class", label, "EM iterations:", len(pca_model.log_likelihoods))
    if len(pca_model.log_likelihoods) == pca_model.iterations:
        print("Iteration limit reached; convergence is not guaranteed.")

pca_val_scores = np.column_stack([pca_model.score_samples(X_val_pca) for pca_model in pca_models])
gmm_pca_val_predictions = classes[np.argmax(pca_val_scores + np.log(pca_priors), axis=1)]

pca_test_scores = np.column_stack([pca_model.score_samples(X_test_pca) for pca_model in pca_models])
gmm_pca_test_predictions = classes[np.argmax(pca_test_scores + np.log(pca_priors), axis=1)]

# Confusion matrix
gmm_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_pca_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_pca_confusion)

Class 0 EM iterations: 22
Class 1 EM iterations: 45
Class 7 EM iterations: 23
Validation accuracy: 0.9947767041002873
Test accuracy: 0.9968183264397072
Test confusion matrix:
 [[ 980    0    0]
 [   0 1135    0]
 [   2    8 1018]]


### **KNN with PCA**

In [25]:
knn_pca = KNN(neighbors=5)
knn_pca.fit(X_train_pca, y_train)
knn_pca_val_predictions = knn_pca.predict(X_val_pca)
knn_pca_test_predictions = knn_pca.predict(X_test_pca)

# Confusion matrix
knn_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_pca_test_predictions == y_test))
print("Test confusion matrix:\n", knn_pca_confusion)

Validation accuracy: 0.9963436928702011
Test accuracy: 0.9914094813872096
Test confusion matrix:
 [[ 976    2    2]
 [   0 1135    0]
 [   1   22 1005]]


### **LDA with PCA**

In [26]:
lda_pca = LDA()
lda_pca.fit(X_train_pca, y_train)
lda_pca_val_predictions = lda_pca.predict(X_val_pca)
lda_pca_test_predictions = lda_pca.predict(X_test_pca)

# Confusion matrix
lda_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        lda_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (lda_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(lda_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(lda_pca_test_predictions == y_test))
print("Test confusion matrix:\n", lda_pca_confusion)

Validation accuracy: 0.980673805171063
Test accuracy: 0.9844097995545658
Test confusion matrix:
 [[ 972    1    7]
 [   0 1132    3]
 [   3   35  990]]
